[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/08_dashboard.ipynb)

# Sitzung 8 — Alles zusammengesetzt: das Insight-Dashboard

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 25.11.2026 · Meilenstein**

In Sitzung 1 habt ihr gesehen, wohin die Reise geht: von hunderten Bewertungen zu einem Bild, das eine Managerin in zwei Minuten versteht. Seitdem habt ihr jeden Baustein einzeln gebaut. Heute setzen wir sie **zusammen**, zum ersten Mal von den Rohdaten bis zum fertigen Dashboard mit einer Zusammenfassung durch das LLM.

Heute gibt es **keinen neuen Stoff**. Alles, was vorkommt, kennt ihr. Neu ist nur, dass es zusammen läuft. Und dieses Notebook ist die **Vorlage für eure Projektarbeit**: Dieselbe Pipeline, auf eure eigenen Daten angewendet.

**Ablauf**
1. **Die Pipeline auf einen Blick (Abschnitt 1):** welcher Baustein aus welcher Sitzung kommt.
2. **Analysieren (Abschnitt 2):** jede Bewertung mit dem LLM einordnen, mit Schema und Korrektur aus Sitzung 6, parallel aus Sitzung 3.
3. **Wie gut ist die Pipeline? (Abschnitt 3):** mit der Wahrheit vergleichen, einzeln und im Aggregat.
4. **Kennzahlen und Dashboard (Abschnitt 4 und 5):** die Antworten aus Sitzung 7, auf einer Seite.
5. **Die Zusammenfassung (Abschnitt 6):** das LLM schreibt, wir prüfen.
6. **Eure Aufgabe (Abschnitt 7):** ein eigener Ausschnitt, und ein Plausibilitätscheck.

Heute brauchen wir wieder den **API-Schlüssel** (Colab Secrets, wie gewohnt).

### Rückblick auf Sitzung 7

```text
1. „Mehr Beschwerden im Monat X"       -> Anteil statt Anzahl: die Gesamtmenge schwankt
2. Monat mit wenigen Bewertungen       -> n und Intervall mitzeigen
3. Sterne als billiges Signal          -> zeigen den Trend, nicht das Warum; fehlende Sterne verzerren
4. „Nach X schlechter, also wegen X"   -> Folgerungen prüfen: Wann beginnt es? Wo zeigt es sich?
5. Ein Segment fällt auf               -> kann Zufall sein; mit neuen Daten bestätigen
6. Ergebnis für die Managerin          -> eine Seite: Antwort, wichtigste Zahl, was nicht belegt ist
```

In Sitzung 7 haben wir die Stimmung und Themen aus der **Wahrheit** genommen, so als hätte die Pipeline fehlerfrei gearbeitet. Heute setzen wir die **echte** Pipeline davor. Die spannende Frage: Bleiben die Antworten an Lena dieselben, wenn das LLM Fehler macht?

## 0. Setup und Daten

**Was passiert in der nächsten Zelle?** Bibliotheken laden, `radar.py` holen, das Modell einstellen und den bereinigten Datensatz aus Sitzung 4 laden. Ohne Schlüssel läuft alles mit dem Mock, die Ergebnisse sehen dann aber deutlich schlechter aus.

In [ ]:
!pip install anthropic --quiet

import json, re, time, urllib.request, importlib
from concurrent.futures import ThreadPoolExecutor
from typing import Literal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
from pydantic import BaseModel, Field, ValidationError
pd.set_option("display.max_colwidth", 90)

URL = "https://raw.githubusercontent.com/kevisback/bda-course/session-08/wise2627/notebooks/radar.py"
try:
    urllib.request.urlretrieve(URL, "radar.py")
except Exception as e:
    print("Download fehlgeschlagen, nutze lokale radar.py:", e)
import radar
importlib.reload(radar)
radar.setup(backend="anthropic", model="claude-haiku-4-5")

df, wahrheit = radar.lade_sauber()
THEMEN = radar.THEMEN
print(df.shape)
df.head()

**Wie lese ich das?** 279 Bewertungen, bereinigt wie in Sitzung 4: keine Duplikate, kein Müll, Sterne und Datum in einheitlichem Format. Sterne und Datum fehlen bei manchen Bewertungen. Das ist der Ausgangspunkt. Alles Weitere berechnen wir heute selbst.

## 1. Die Pipeline auf einen Blick

Jeder Kasten ist ein Baustein, den ihr schon gebaut habt:

```text
Rohdaten ──> bereinigen ──> analysieren ──────────> prüfen ──> Kennzahlen ──> Dashboard ──> Zusammenfassung
             (Sitzung 4)    (LLM: Stimmung,         (Sitzung    (Sitzung 7)    (heute)       (LLM, heute)
                             Themen, dringend;       3 und 5:                                    │
                             Schema + Korrektur      Vergleich                                   ▼
                             aus Sitzung 6,          mit der                              Plausibilitäts-
                             parallel aus            Wahrheit)                            check (Sitzung 9)
                             Sitzung 3)
```

Zwei Dinge sind wichtig an diesem Bild:
- **Das LLM kommt zweimal vor:** einmal für jede einzelne Bewertung (viele kleine, gut prüfbare Aufgaben), einmal am Ende für den Text (eine große, schwer prüfbare Aufgabe).
- **Prüfen ist ein eigener Schritt.** Eine Pipeline ohne Prüfschritt liefert schöne Grafiken, von denen niemand weiß, ob sie stimmen.

## 2. Analysieren: jede Bewertung einordnen

Wir nehmen den Vertrag aus Sitzung 6, unverändert: Stimmung, Themen mit Polarität, eine kurze Begründung, und `dringend`.

**Was passiert in der nächsten Zelle?** Das Schema als pydantic-Klasse und der Prompt, beides wie in Sitzung 6.

In [ ]:
class Nennung(BaseModel):
    thema: Literal[tuple(THEMEN)]                      # nur Themen aus dem Katalog
    polaritaet: Literal["positiv", "negativ"]

class Analyse(BaseModel):
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    themen: list[Nennung]
    begruendung: str = Field(max_length=200)
    dringend: bool

ANALYSE_PROMPT = """Du analysierst eine Produktbewertung zu kabellosen Ohrhörern.

Antworte ausschließlich mit einem JSON-Objekt mit genau diesen Feldern:
- "sentiment": "positive", "negative", "mixed" oder "neutral"
- "themen": Liste von {"thema": <Thema>, "polaritaet": "positiv" oder "negativ"}
   Erlaubte Themen: """ + ", ".join(THEMEN) + """
- "begruendung": höchstens 20 Wörter
- "dringend": true, wenn ein Defekt oder ein Problem gemeldet wird, um das sich jemand kümmern sollte, sonst false"""

print(ANALYSE_PROMPT)

**Wie lese ich das?** Das ist genau der Prompt aus Sitzung 6. Für eine Pipeline ist das wichtig: Prompt und Schema sind **festgelegt und versioniert**, nicht für jeden Lauf neu formuliert. Wer den Prompt ändert, muss die Prüfung in Abschnitt 3 neu machen.

**Was passiert in der nächsten Zelle?** Fragen, prüfen, korrigieren, wie in Sitzung 6, nur ohne die künstlichen Störungen. Eine Funktion `analysiere(text)` liefert am Ende ein `Analyse`-Objekt oder `None`.

In [ ]:
def pruefe(roh):
    """-> (Analyse oder None, Fehlermeldung oder None). Wie in Sitzung 6."""
    m = re.search(r"\{.*\}", roh or "", re.DOTALL)                 # Stufe 1: herausschneiden
    if not m:
        return None, "Kein JSON-Objekt gefunden."
    text = re.sub(r",\s*([}\]])", r"\1", m.group(0))               # Stufe 2a: Komma reparieren
    try:
        daten = json.loads(text)                                    # Stufe 2: einlesen
    except json.JSONDecodeError as e:
        return None, f"Kein gültiges JSON: {e.msg}"
    try:
        return Analyse.model_validate(daten), None                  # Stufe 3: Vertrag prüfen
    except ValidationError as e:
        return None, "; ".join(f"{'.'.join(str(x) for x in f['loc'])}: {f['msg']}" for f in e.errors())

def frage_modell(text, korrektur=None):
    user = f'Bewertung:\n"""{text}"""'
    if korrektur:
        alt, fehler = korrektur
        user += (f"\n\nDeine letzte Antwort war ungültig.\nAntwort: {alt}\nFehler: {fehler}\n"
                 "Antworte nur mit dem korrigierten JSON-Objekt.")
    if radar.BACKEND == "mock":                                     # Mock: aus Stichwörtern zusammenbauen
        themen = radar._mock_themen(text)
        s = json.loads(radar.call_llm(user, temperature=0)["text"])["sentiment"]
        return json.dumps({"sentiment": s, "themen": themen, "begruendung": "Mock",
                           "dringend": any(t["polaritaet"] == "negativ" for t in themen)}, ensure_ascii=False)
    return radar.mit_retry(radar.call_llm, user, system=ANALYSE_PROMPT,
                           temperature=0, max_tokens=400)["text"]   # 0 = Cache beim zweiten Lauf

def analysiere(text, max_korrekturen=2):
    """-> (Analyse oder None, Anzahl Anfragen)"""
    roh = frage_modell(text)
    for anfrage in range(1, max_korrekturen + 2):
        analyse, fehler = pruefe(roh)
        if analyse:
            return analyse, anfrage
        if anfrage <= max_korrekturen:
            roh = frage_modell(text, korrektur=(roh, fehler))
    return None, anfrage

beispiel = df["text"].iloc[3]
a, n = analysiere(beispiel)
print("Bewertung:", beispiel, "\n")
print(a.model_dump_json(indent=2) if a else "aufgegeben", f"\n\n({n} Anfrage(n))")

**Wie lese ich das?** Eine Bewertung rein, ein geprüftes Objekt raus. Die drei Funktionen sind dieselben wie in Sitzung 6, nur kürzer, weil wir heute nichts künstlich stören. Zwei Kleinigkeiten sind neu:
- `radar.mit_retry` wiederholt bei Überlast oder Rate-Limit automatisch (Sitzung 3).
- `temperature=0` sorgt dafür, dass ein zweiter Lauf aus dem Cache kommt und nichts kostet.

**Was passiert in der nächsten Zelle?** Jetzt alle 279 Bewertungen. Nacheinander würde das mehrere Minuten dauern, deshalb schicken wir **acht Anfragen gleichzeitig** (Sitzung 3). Danach zählen wir Zeit, Anfragen und Kosten.

In [ ]:
radar.reset_verbrauch()
start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:                    # 8 Anfragen gleichzeitig
    ergebnisse = list(pool.map(analysiere, df["text"]))
dauer = time.time() - start

df["analyse"] = [a for a, _ in ergebnisse]
df["anfragen"] = [n for _, n in ergebnisse]
ok = df["analyse"].notna()

print(f"{ok.sum()} von {len(df)} Bewertungen gültig analysiert, {(~ok).sum()} aufgegeben")
print(f"davon {(df['anfragen'] > 1).sum()} erst nach Korrektur gültig")
print(f"Dauer: {dauer:.0f} s | Verbrauch: {radar.verbrauch()}")

**Wie lese ich das?** Fast alle Bewertungen sollten im ersten Versuch gültig sein; ein paar brauchen eine Korrektur, aufgegeben wird selten etwas. Die Kosten liegen bei Haiku im Bereich von Cent.

Der Verbrauch zeigt auch den Cache: Aufrufe plus Cache-Treffer ergeben alle Anfragen (279 Bewertungen plus Korrekturen). Cache-Treffer gibt es schon beim ersten Lauf, weil manche Bewertungen wortgleich sind. Dieselbe Anfrage wird nur einmal bezahlt. Führt die Zelle ein zweites Mal aus: Dann kommt alles aus dem Cache, die Zelle ist in einer Sekunde fertig und der Verbrauch ist null.

Aufgegebene Bewertungen fallen **nicht** still weg: Wir behalten sie in der Tabelle und zählen sie mit. Im Bericht muss stehen, auf wie vielen Bewertungen die Zahlen beruhen.

**Was passiert in der nächsten Zelle?** Wir schauen uns die Bewertung an, die eine Korrektur brauchte: die erste Antwort des Modells, die Meldung des Prüfers und die korrigierte Antwort.

In [ ]:
# Die Bewertung(en), die eine Korrektur brauchten, Schritt für Schritt
korrigiert = df[df["anfragen"] > 1]
print(f"{len(korrigiert)} Bewertung(en) brauchten eine Korrektur.\n")

if len(korrigiert):
    text = korrigiert["text"].iloc[0]
    print("Bewertung:\n  ", text, "\n")
    roh = frage_modell(text)                                   # kommt aus dem Cache, kostet nichts
    for runde in range(1, korrigiert["anfragen"].iloc[0]):
        analyse, fehler = pruefe(roh)
        print(f"① Antwort {runde}:\n   {roh}\n")
        print(f"② Prüfer meldet:\n   {fehler}\n")
        roh = frage_modell(text, korrektur=(roh, fehler))      # die Korrektur-Anfrage, ebenfalls aus dem Cache
    analyse, fehler = pruefe(roh)
    print(f"③ Antwort nach der Korrektur:\n   {roh}\n")
    print("④ Prüfer:", "✅ gültig" if analyse else "❌ " + fehler)
else:
    print("Diesmal war jede Antwort im ersten Versuch gültig.")

**Wie lese ich das?** Dasselbe Muster wie in Sitzung 6, nur diesmal ohne künstliche Störung: Das Modell hat von sich aus eine ungültige Antwort geliefert. Der Prüfer hat sie erkannt und genau gesagt, was nicht stimmt, und das Modell hat mit dieser Fehlermeldung korrigiert. Bei 279 Bewertungen passiert das selten, aber ohne Prüfer wäre dieser Fehler still in die Kennzahlen gelaufen.

Warum kostet diese Zelle nichts? Alle Anfragen liefen mit `temperature=0`. Dieselbe Anfrage liefert dann dieselbe Antwort, und die kommt aus dem Cache.

**Was passiert in der nächsten Zelle?** Was würde dieselbe Pipeline für einen großen Shop kosten? Wir rechnen die Kosten dieses Laufs auf 10.000 und 100.000 Bewertungen hoch.

In [ ]:
v = radar.verbrauch()
if v["aufrufe"]:
    pro_bewertung = v["usd"] / len(df)
    zeit_pro_bewertung = dauer / len(df)
    for menge in [len(df), 10_000, 100_000]:
        print(f"{menge:>7} Bewertungen:  ca. {menge * pro_bewertung:7.2f} USD,  "
              f"ca. {menge * zeit_pro_bewertung / 60:5.1f} Minuten (bei 8 parallelen Anfragen)")
else:
    print("Alles kam aus dem Cache: Kosten 0. Für die Hochrechnung den Cache leeren (radar.CACHE.clear()) und Abschnitt 2 neu ausführen.")

**Wie lese ich das?** Eine einfache Dreisatz-Rechnung, aber eine, die jede Managerin als Erstes fragt. Bei Haiku kostet die Analyse einer Bewertung Bruchteile eines Cents; auch 100.000 Bewertungen bleiben bezahlbar. Die Zeit wird eher zum Thema, und dort helfen mehr parallele Anfragen, bis das Rate-Limit greift (Sitzung 3). Kosten und Grenzen im Großen sind Thema von Sitzung 11.

**Was passiert in der nächsten Zelle?** Wir packen die Analyse-Objekte in einfache Spalten aus, damit wir mit pandas weiterrechnen können wie in Sitzung 7.

In [ ]:
df["sentiment_llm"] = [a.sentiment if a else None for a in df["analyse"]]
df["themen_llm"] = [[(n.thema, n.polaritaet) for n in a.themen] if a else [] for a in df["analyse"]]
df["dringend"] = [a.dringend if a else None for a in df["analyse"]]
df["begruendung"] = [a.begruendung if a else None for a in df["analyse"]]

# aus Sitzung 7: Wahrheit zum Vergleich (gibt es nur im Kurs)
df["sentiment_wahr"] = wahrheit.loc[df["review_id"], "true_sentiment"].values
df["themen_wahr"] = [radar.wahre_themen(wahrheit.loc[r, "text_wahr"]) for r in df["review_id"]]
df["quartal"] = df["datum"].dt.to_period("Q")

df[["text", "sentiment_llm", "themen_llm", "dringend", "begruendung"]].head(8)

**Wie lese ich das?** Ab hier ist jede Bewertung eine Zeile mit festen Feldern, genau das Format, mit dem wir in Sitzung 7 gerechnet haben. Die Spalten `sentiment_wahr` und `themen_wahr` gibt es nur, weil wir die Daten selbst erzeugt haben. In euren Projekten fehlt diese Wahrheit; dort braucht ihr eine **Stichprobe, die ihr selbst annotiert** (Sitzung 3 und 5).

## 3. Wie gut ist die Pipeline?

Bevor wir irgendetwas zeigen, prüfen wir. Drei Fragen, in dieser Reihenfolge:
1. Wie oft stimmt die **Stimmung**?
2. Wie gut werden die **Themen** erkannt?
3. Und am wichtigsten: Ändern die Fehler die **Antworten an Lena**?

**Was passiert in der nächsten Zelle?** Eine Kreuztabelle wie in Sitzung 3: Zeilen = Wahrheit, Spalten = LLM.

In [ ]:
REIHENFOLGE = ["positive", "mixed", "neutral", "negative"]
FARBEN = {"positive": "#16a34a", "mixed": "#f59e0b", "neutral": "#9ca3af", "negative": "#dc2626"}

geprueft = df[ok]
treffer = (geprueft["sentiment_llm"] == geprueft["sentiment_wahr"]).mean()
print(f"Stimmung richtig: {treffer:.0%} von {len(geprueft)} Bewertungen\n")
pd.crosstab(geprueft["sentiment_wahr"], geprueft["sentiment_llm"],
            rownames=["wahr"], colnames=["LLM"]).reindex(index=REIHENFOLGE, columns=REIHENFOLGE, fill_value=0)

**Wie lese ich das?** Auf der Diagonalen stehen die Treffer. Die meisten Fehler liegen erfahrungsgemäß zwischen *mixed* und den Nachbarn: Ist „Klang top, Akku schwach" gemischt oder eher negativ? Das ist dieselbe Unschärfe, die wir in Sitzung 1 und 3 diskutiert haben, und kein Fehler der Pipeline-Technik.

**Was passiert in der nächsten Zelle?** Fünf Bewertungen, bei denen LLM und Wahrheit nicht übereinstimmen, mit der **Begründung** des Modells (Sitzung 6).

In [ ]:
abweichend = geprueft[geprueft["sentiment_llm"] != geprueft["sentiment_wahr"]].drop_duplicates("text")
with pd.option_context("display.max_colwidth", None):
    display(abweichend[["text", "sentiment_wahr", "sentiment_llm", "begruendung"]].head(5))

**Wie lese ich das?** Lest die Begründungen: Oft ist die Entscheidung des Modells nachvollziehbar, und man könnte über die Wahrheit streiten (Sitzung 1: Ist unsere Wahrheit zu streng?). Manchmal liegt das Modell klar daneben, etwa bei Ironie. Genau dafür gibt es die Begründung: Ohne sie sähe man nur, *dass* es abweicht, nicht *warum*.

**Was passiert in der nächsten Zelle?** Die Themen, mit Precision, Recall und F1 aus Sitzung 5, über alle Themen zusammen.

In [ ]:
def themen_menge(liste):
    return {t for t, _ in liste}

tp = sum(len(themen_menge(l) & themen_menge(w)) for l, w in zip(geprueft["themen_llm"], geprueft["themen_wahr"]))
fp = sum(len(themen_menge(l) - themen_menge(w)) for l, w in zip(geprueft["themen_llm"], geprueft["themen_wahr"]))
fn = sum(len(themen_menge(w) - themen_menge(l)) for l, w in zip(geprueft["themen_llm"], geprueft["themen_wahr"]))
precision, recall = tp / (tp + fp), tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(f"Themen: Precision {precision:.0%} | Recall {recall:.0%} | F1 {f1:.0%}")
print(f"({tp} richtig gefunden, {fp} Fehlalarme, {fn} übersehen)")

**Wie lese ich das?** Precision: Von den Themen, die das LLM nennt, wie viele stimmen? Recall: Von den echten Themen, wie viele findet es? Beide Werte sollten deutlich über dem Stichwort-Verfahren aus Sitzung 5 liegen.

**Was passiert in der nächsten Zelle?** Und `dringend`? Dafür haben wir keine Wahrheit. Wir können aber prüfen, ob es **zusammenpasst**: Wie oft ist eine Bewertung dringend, je nach ihrer (wahren) Stimmung?

In [ ]:
dringend_tab = geprueft.groupby("sentiment_wahr")["dringend"].agg(["mean", "count"]).reindex(REIHENFOLGE)
dringend_tab.columns = ["Anteil dringend", "Bewertungen"]
dringend_tab.style.format({"Anteil dringend": "{:.0%}"})

**Wie lese ich das?** Wenn `dringend` sinnvoll ist, sollten negative Bewertungen oft dringend sein, positive fast nie. Gemischte liegen dazwischen. Das ist kein Beweis, dass jeder einzelne Wert stimmt, aber ein **Plausibilitätscheck**: Wäre ein Drittel der positiven Bewertungen dringend, wüssten wir, dass etwas mit dem Feld nicht stimmt. Erinnert euch an Sitzung 6: `dringend` ist unscharf definiert. Für die Projektarbeit gilt: Jedes Feld ohne Wahrheit braucht mindestens so einen Check.

Einzelne Fehler sind aber nicht das Entscheidende. Lena liest keine einzelnen Bewertungen, sie liest **Anteile**.

**Was passiert in der nächsten Zelle?** Die wichtigste Prüfung: der Anteil negativer Bewertungen pro Quartal, einmal mit der Wahrheit (wie in Sitzung 7), einmal mit der Pipeline.

In [ ]:
mit_datum = df[ok & df["datum"].notna()]
vergleich = pd.DataFrame({
    "wahr (Sitzung 7)": (mit_datum["sentiment_wahr"] == "negative").groupby(mit_datum["quartal"]).mean(),
    "Pipeline (heute)": (mit_datum["sentiment_llm"] == "negative").groupby(mit_datum["quartal"]).mean(),
})
ax = vergleich.plot(kind="bar", color=["#9ca3af", "#dc2626"], figsize=(7, 3.5), width=0.75)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}")); ax.set_ylim(0, 1)
ax.set_title("Anteil negativer Bewertungen pro Quartal: Wahrheit gegen Pipeline"); ax.set_xlabel("")
ax.legend(frameon=False, loc="upper left"); ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0); plt.tight_layout(); plt.show()
abweichung = (vergleich["Pipeline (heute)"] - vergleich["wahr (Sitzung 7)"]).abs().max()
print(f"Größte Abweichung zwischen Pipeline und Wahrheit: {abweichung:.0%}-Punkte")
vergleich.style.format("{:.0%}")

**Wie lese ich das?** Liegen die Balkenpaare nah beieinander, gibt die Pipeline Lena **dieselbe Antwort** wie die Wahrheit: Es wird deutlich schlechter. Kleine Abweichungen sind normal; entscheidend ist, ob der Trend und die Größenordnung stimmen.

Das ist die Lehre aus Sitzung 3, jetzt in der fertigen Pipeline: Einzelfehler sind unvermeidlich. Ob sie schaden, sieht man erst im **Aggregat**, und das muss man messen, nicht annehmen.

> **Nicht:** „Das LLM ist gut, also stimmt das Dashboard."
> **Sondern:** „Wir haben verglichen: Auf der Ebene, auf der Entscheidungen fallen, weicht die Pipeline um höchstens so viele Prozentpunkte ab" (die Zahl unter der Grafik).

**Was passiert in der nächsten Zelle?** Dieselbe Frage für die Themen: Hätte Lena mit der Pipeline dieselbe **Rangliste der Kritikpunkte** bekommen wie mit der Wahrheit?

In [ ]:
def kritik_rangliste(spalte):
    zaehler = pd.Series([t for l in geprueft[spalte] for t, p in l if p == "negativ"]).value_counts()
    return zaehler.rename_axis("thema")

rang = pd.DataFrame({"wahr": kritik_rangliste("themen_wahr"), "Pipeline": kritik_rangliste("themen_llm")}).fillna(0).astype(int)
rang["Platz wahr"] = rang["wahr"].rank(ascending=False, method="min").astype(int)
rang["Platz Pipeline"] = rang["Pipeline"].rank(ascending=False, method="min").astype(int)
rang.sort_values("wahr", ascending=False)

**Wie lese ich das?** Links die Anzahl negativer Nennungen laut Wahrheit, daneben laut Pipeline, rechts die Plätze. Die Pipeline findet insgesamt etwas weniger Nennungen (Recall unter 100 %), entscheidend ist aber die **Reihenfolge**: Stehen dieselben Themen oben? Wenn zwei Themen nur wenige Nennungen auseinanderliegen, können sie leicht die Plätze tauschen. Im Dashboard sollte dann nicht stehen „Platz 1 ist X", sondern „X und Y sind die häufigsten Kritikpunkte".

## 4. Kennzahlen

Ab hier rechnen wir nur noch mit den Pipeline-Ergebnissen, so wie es in der Praxis wäre. Alle Kennzahlen kommen aus **einer** Funktion. Das hat einen Grund: Für jeden Ausschnitt (ein Quartal, eine Quelle, ein Produkt) rufen wir dieselbe Funktion auf, und für eure Projektarbeit auch.

**Was passiert in der nächsten Zelle?** `kennzahlen(teil)` berechnet alles, was Lena wissen will, für eine beliebige Teilmenge der Bewertungen.

In [ ]:
def kennzahlen(teil):
    """Alle Zahlen fürs Dashboard, für eine beliebige Teilmenge df[...]."""
    g = teil[teil["analyse"].notna()]
    n = len(g)
    k_neg = int((g["sentiment_llm"] == "negative").sum())
    von, bis = radar.wilson(k_neg, n)

    quartale = (g.dropna(subset=["datum"])
                 .assign(neg=lambda x: x["sentiment_llm"] == "negative")
                 .groupby("quartal")["neg"].agg(["sum", "count"]))

    nennungen = pd.DataFrame([{"thema": t, "polaritaet": p} for l in g["themen_llm"] for t, p in l],
                             columns=["thema", "polaritaet"])
    kritik = nennungen[nennungen["polaritaet"] == "negativ"]["thema"].value_counts()
    lob = nennungen[nennungen["polaritaet"] == "positiv"]["thema"].value_counts()

    return {
        "bewertungen": len(teil), "analysiert": n, "aufgegeben": len(teil) - n,
        "zeitraum": f"{g['datum'].min():%d.%m.%Y} bis {g['datum'].max():%d.%m.%Y}",
        "verteilung": g["sentiment_llm"].value_counts().reindex(REIHENFOLGE, fill_value=0),
        "anteil_negativ": k_neg / n, "intervall_negativ": (von, bis),
        "quartale": quartale,
        "kritik": kritik, "lob": lob,
        "anteil_dringend": g["dringend"].mean(),
        "sterne_mittel": g["sterne"].astype("float").mean(),
        "sterne_fehlt": g["sterne"].isna().mean(),
    }

k = kennzahlen(df)
for name in ["bewertungen", "analysiert", "aufgegeben", "zeitraum", "anteil_negativ",
             "intervall_negativ", "anteil_dringend", "sterne_mittel", "sterne_fehlt"]:
    print(f"{name:18} {k[name]}")
print("\nhäufigste Kritik:\n", k["kritik"].head(3).to_string())

**Wie lese ich das?** Ein Dictionary mit allen Zahlen. Noch nicht schön, aber vollständig, und jede Zahl ist nachvollziehbar: Sie kommt aus den Spalten, die wir oben geprüft haben. Drei Dinge stehen bewusst mit drin, weil sie in Sitzung 7 wichtig waren:
- **`aufgegeben`:** auf wie vielen Bewertungen die Zahlen beruhen,
- **`intervall_negativ`:** wie sicher der Anteil ist,
- **`sterne_fehlt`:** wie verlässlich der Sterne-Durchschnitt ist.

## 5. Das Dashboard

Ein Dashboard soll eine Frage in Sekunden beantworten. Deshalb: **sauber statt verspielt.** Vier Felder, jedes beantwortet eine Frage von Lena:

```text
┌────────────────────────────┬────────────────────────────┐
│ Wo stehen wir?             │ Wird es schlechter?        │
│ (Kennzahlen in Zahlen)     │ (Anteil negativ pro        │
│                            │  Quartal, mit Intervall)   │
├────────────────────────────┼────────────────────────────┤
│ Worüber wird geklagt?      │ Was läuft gut?             │
│ (Themen in Beschwerden)    │ (Themen in Lob)            │
└────────────────────────────┴────────────────────────────┘
```

**Was passiert in der nächsten Zelle?** Eine Funktion `dashboard(teil, titel)`, die genau dieses Bild zeichnet.

In [ ]:
def dashboard(teil, titel):
    k = kennzahlen(teil)
    fig, achsen = plt.subplots(2, 2, figsize=(12, 7))
    (oben_l, oben_r), (unten_l, unten_r) = achsen
    fig.suptitle(f"{titel}  ·  {k['analysiert']} Bewertungen  ·  {k['zeitraum']}", fontsize=13, x=0.01, ha="left")

    # 1. Wo stehen wir?
    oben_l.axis("off")
    von, bis = k["intervall_negativ"]
    kacheln = [("negativ", f"{k['anteil_negativ']:.0%}", f"Intervall {von:.0%} bis {bis:.0%}"),
               ("dringend", f"{k['anteil_dringend']:.0%}", "Defekte und Probleme"),
               ("Ø Sterne", f"{k['sterne_mittel']:.1f}".replace(".", ","), f"{k['sterne_fehlt']:.0%} ohne Sterne")]
    for i, (name, wert, zusatz) in enumerate(kacheln):
        y = 0.82 - i * 0.33
        oben_l.text(0.02, y, wert, fontsize=26, weight="bold", color="#111827", va="center")
        oben_l.text(0.40, y + 0.06, name, fontsize=11, color="#111827", va="center")
        oben_l.text(0.40, y - 0.06, zusatz, fontsize=9, color="#6b7280", va="center")
    if k["aufgegeben"]:
        oben_l.text(0.02, -0.08, f"{k['aufgegeben']} Bewertungen nicht auswertbar", fontsize=8, color="#6b7280")
    oben_l.set_title("Wo stehen wir?", loc="left")

    # 2. Wird es schlechter?
    q = k["quartale"]
    if len(q):
        x = q.index.astype(str)
        anteil = q["sum"] / q["count"]
        band = [radar.wilson(s, n) for s, n in zip(q["sum"], q["count"])]
        oben_r.fill_between(x, [b[0] for b in band], [b[1] for b in band], color="#dc2626", alpha=0.15)
        oben_r.plot(x, anteil, marker="o", color="#dc2626")
        for xi, n in zip(x, q["count"]):
            oben_r.annotate(f"n = {n}", (xi, 0.04), ha="center", fontsize=8, color="#6b7280")
    oben_r.set_ylim(0, 1); oben_r.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    oben_r.set_title("Wird es schlechter? (Anteil negativ pro Quartal)", loc="left")

    # 3. und 4. Themen
    for ax, reihe, farbe, frage in [(unten_l, k["kritik"], "#dc2626", "Worüber wird geklagt?"),
                                    (unten_r, k["lob"], "#16a34a", "Was läuft gut?")]:
        top = reihe.head(5)[::-1]
        ax.barh(top.index, top.values, color=farbe, alpha=0.85)
        for i, v in enumerate(top.values):
            ax.annotate(str(v), (v, i), xytext=(3, 0), textcoords="offset points", va="center", fontsize=9)
        ax.set_title(f"{frage} (Nennungen)", loc="left")

    for ax in achsen.flat:
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    return k

k_alle = dashboard(df, "Review Radar · Nimbus Q2")

**Wie lese ich das?** Oben links die drei wichtigsten Zahlen, jede mit ihrer Einschränkung darunter. Oben rechts der Trend, mit Intervall und n, wie in Sitzung 7. Unten, worüber geklagt und was gelobt wird, gezählt als Nennungen.

Was bewusst **fehlt**: Farbverläufe, 3D, Tachometer, zehn Kennzahlen. Jedes zusätzliche Element kostet Aufmerksamkeit. Ein guter Test: Kann jemand in zehn Sekunden sagen, was die wichtigste Aussage ist? Hier etwa: *Es wird deutlich schlechter, und die Kritik verteilt sich auf mehrere Themen.*

Zum Vergleich mit Sitzung 1: Damals habt ihr ein fertiges Dashboard gesehen. Jetzt wisst ihr, was in jeder Zahl steckt, und wo sie falsch sein könnte.

## 6. Die Zusammenfassung

Lena liest das Dashboard, ihr Chef liest nur **einen Absatz**. Den lassen wir das LLM schreiben. Aber nicht aus den Bewertungen, sondern aus den **Kennzahlen**: Das LLM soll formulieren, nicht rechnen.

```text
Bewertungen ──> Pipeline ──> Kennzahlen ──> Fakten (JSON) ──> LLM ──> Absatz
                                              ▲                        │
                                              └── Zahlen-Check ◄───────┘
```

**Was passiert in der nächsten Zelle?** Wir machen aus den Kennzahlen eine kurze Liste von **Fakten**, mit gerundeten Zahlen. Nur diese Fakten bekommt das LLM.

In [ ]:
def fakten(k, titel):
    q = k["quartale"]
    return {
        "titel": titel,
        "zeitraum": k["zeitraum"],
        "bewertungen_ausgewertet": k["analysiert"],
        "nicht_auswertbar": k["aufgegeben"],
        "anteil_negativ_prozent": round(100 * k["anteil_negativ"]),
        "intervall_negativ_prozent": [round(100 * x) for x in k["intervall_negativ"]],
        "anteil_negativ_pro_quartal_prozent": {str(i): round(100 * s / n) for i, s, n in zip(q.index, q["sum"], q["count"])},
        "bewertungen_pro_quartal": {str(i): int(n) for i, n in zip(q.index, q["count"])},
        "haeufigste_kritik_nennungen": k["kritik"].head(3).to_dict(),
        "haeufigstes_lob_nennungen": k["lob"].head(3).to_dict(),
        "anteil_dringend_prozent": round(100 * k["anteil_dringend"]),
        "sterne_mittel": round(k["sterne_mittel"], 1),
        "anteil_ohne_sterne_prozent": round(100 * k["sterne_fehlt"]),
    }

f_alle = fakten(k_alle, "Nimbus Q2, alle Bewertungen")
print(json.dumps(f_alle, indent=2, ensure_ascii=False))

**Wie lese ich das?** Das ist alles, was das LLM erfährt. Keine einzige Bewertung, nur Zahlen, die wir geprüft haben. Damit hat das LLM gar nicht die Gelegenheit, aus einem auffälligen Einzelfall eine Tendenz zu machen. Erfinden kann es trotzdem, deshalb prüfen wir gleich.

**Was passiert in der nächsten Zelle?** Der Prompt für die Zusammenfassung, und der Aufruf.

In [ ]:
SUMMARY_PROMPT = """Du schreibst für eine Produktmanagerin eine Zusammenfassung von Kundenbewertungen.

Regeln:
- Verwende ausschließlich die Zahlen aus den Fakten. Rechne nichts Neues aus, erfinde nichts dazu.
- 4 bis 5 Sätze, sachlich, auf Deutsch, ohne Aufzählungszeichen.
- Nenne den Trend, die wichtigsten Kritikpunkte und was gut läuft.
- Nenne mindestens eine Einschränkung (z. B. Unsicherheit, fehlende Sterne, nicht auswertbare Bewertungen).
- Keine Vermutungen über Ursachen."""

def zusammenfassung(f):
    if radar.BACKEND == "mock":                                     # ohne Schlüssel: Textbaustein
        kritik = ", ".join(f["haeufigste_kritik_nennungen"])
        return (f"(Mock) Von {f['bewertungen_ausgewertet']} Bewertungen sind {f['anteil_negativ_prozent']} % negativ. "
                f"Am häufigsten kritisiert: {kritik}.")
    antwort = radar.mit_retry(radar.call_llm, "Fakten:\n" + json.dumps(f, ensure_ascii=False, indent=1),
                              system=SUMMARY_PROMPT, temperature=0, max_tokens=500)
    return antwort["text"].strip()

text_alle = zusammenfassung(f_alle)
display(Markdown("**Zusammenfassung:**\n\n" + text_alle))

**Wie lese ich das?** Ein flüssiger Absatz, der klingt, als hätte ihn jemand aus dem Team geschrieben. Genau das ist die Gefahr: **Er klingt immer überzeugend**, auch wenn eine Zahl nicht stimmt. Der Chef sieht nie die Rohdaten, er sieht nur diesen Absatz.

Die Prompt-Regeln helfen, garantieren aber nichts (Sitzung 6: Ein Prompt ist eine Bitte, kein Vertrag). Also prüfen wir.

**Was passiert in der nächsten Zelle?** Ein einfacher Zahlen-Check: Wir suchen jede Zahl im Text und schauen, ob sie in den Fakten vorkommt.

In [ ]:
def zahlen_check(text, f):
    """Jede Zahl im Text: kommt sie in den Fakten vor?"""
    erlaubt = set()
    def sammle(x):
        if isinstance(x, dict):
            for v in x.values(): sammle(v)
        elif isinstance(x, (list, tuple)):
            for v in x: sammle(v)
        elif isinstance(x, (int, float, np.integer, np.floating)):
            erlaubt.add(round(float(x), 1))
        elif isinstance(x, str):
            for z in re.findall(r"\d+(?:[.,]\d+)?", x):
                erlaubt.add(round(float(z.replace(",", ".")), 1))
    sammle(f)
    zeilen = []
    for z in re.findall(r"\d+(?:[.,]\d+)?", text):
        wert = round(float(z.replace(",", ".")), 1)
        zeilen.append({"Zahl im Text": z, "in den Fakten?": "✅" if wert in erlaubt else "❓ prüfen"})
    return pd.DataFrame(zeilen)

zahlen_check(text_alle, f_alle)

**Wie lese ich das?** ✅ heißt: Diese Zahl steht so in den Fakten. ❓ heißt: Das LLM hat eine Zahl verwendet, die wir ihm **nicht** gegeben haben. Das kann harmlos sein (eine Jahreszahl, „drei Kritikpunkte", eine Rundung), oder es hat gerechnet oder erfunden. Jedes ❓ schaut ein Mensch an.

Und was der Check **nicht** findet: Behauptungen ohne Zahl. „Die Kunden sind vor allem über die App verärgert" kann richtig klingen und trotzdem falsch sein, wenn die App gar nicht Platz 1 der Kritik ist. Solche Sätze prüft man gegen die Fakten, oder gegen die Bewertungen selbst.

**Was passiert in der nächsten Zelle?** Funktioniert der Check überhaupt? Wie in Sitzung 6 testen wir den Prüfer mit einer absichtlich falschen Antwort: Wir hängen einen Satz mit einer erfundenen Zahl an.

In [ ]:
manipuliert = text_alle + " Besonders auffällig: 47 % der Kunden wollen das Produkt zurückgeben."
zahlen_check(manipuliert, f_alle)

**Wie lese ich das?** Die 47 steht nicht in den Fakten, der Check markiert sie mit ❓. Ein Prüfer, den man nie mit einem bekannten Fehler getestet hat, ist kein Prüfer. Gleichzeitig zeigt das Beispiel die Grenze: Hätte der erfundene Satz *keine* Zahl gehabt („Viele Kunden wollen das Produkt zurückgeben"), wäre er unbemerkt durchgerutscht.

**Was passiert in der nächsten Zelle?** Eine kleine Hilfe dafür: `belege(thema)` zeigt Bewertungen, in denen die Pipeline ein Thema als Kritik erkannt hat, mit der Begründung des Modells.

In [ ]:
def belege(thema, polaritaet="negativ", n=4, teil=None):
    """Zeigt Bewertungen, in denen die Pipeline das Thema mit dieser Polarität erkannt hat."""
    teil = df if teil is None else teil
    maske = [(thema, polaritaet) in l for l in teil["themen_llm"]]
    treffer = teil[maske][["datum", "text", "begruendung"]].drop_duplicates("text")
    print(f"{sum(maske)} Bewertungen mit {thema} ({polaritaet}), {min(n, len(treffer))} davon:")
    with pd.option_context("display.max_colwidth", None):
        display(treffer.head(n))

belege(k_alle["kritik"].index[0])

**Wie lese ich das?** Die häufigste Kritik aus dem Dashboard, mit echten Bewertungen dahinter. Passen sie zum Thema? Passt die Begründung des Modells? Genau so prüft man eine Behauptung aus der Zusammenfassung: Zahl in den Fakten nachschlagen, und für das *Worüber* zwei, drei Bewertungen lesen.

> 💡 **Kernpunkt:** Das LLM schreibt den Text, aber **wir** sind für jede Aussage darin verantwortlich.
>
> **Nicht:** „Das hat das LLM zusammengefasst, das wird schon stimmen."
> **Sondern:** Das LLM formuliert aus geprüften Zahlen, und jede Zahl und jede Behauptung im Text lässt sich auf eine Quelle zurückführen.

## 7. Eure Aufgabe

✏️ **Teil 1: ein eigener Ausschnitt.** Lena will das Dashboard nur für die **deutsche App** (`app_de`), oder für ein Quartal eurer Wahl. Weil alles in Funktionen steckt, sind das drei Zeilen.

1. Füllt die Lücken `...`: Wählt einen Ausschnitt von `df` aus.
2. Zeichnet das Dashboard, erzeugt die Fakten und die Zusammenfassung.

Die Lösung steht in der Zelle danach. Erst selbst versuchen.

In [ ]:
# Eure Lösung: Lücken ... füllen
# (try/except nur, damit „Alle ausführen" hier nicht stoppt. Die Lösungszelle darunter braucht es nicht.)
try:
    teil = df[...]                                       # z. B. df["quelle"] == "app_de"
    k_teil = dashboard(teil, ...)                        # ein Titel als Text
    f_teil = fakten(k_teil, ...)
    text_teil = zusammenfassung(f_teil)
    display(Markdown("**Zusammenfassung:**\n\n" + text_teil))
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
teil = df[df["quelle"] == "app_de"]
k_teil = dashboard(teil, "Review Radar · Nimbus Q2 · deutsche App")
f_teil = fakten(k_teil, "Nimbus Q2, deutsche App")
text_teil = zusammenfassung(f_teil)
display(Markdown("**Zusammenfassung:**\n\n" + text_teil))

**Wie lese ich das?** Dasselbe Dashboard, ein anderer Ausschnitt, ohne eine einzige neue Zeile Analyse-Code. Das ist der Vorteil einer Pipeline aus Funktionen. Achtet auf die Zahl der Bewertungen: Bei einem kleinen Ausschnitt werden die Intervalle breit, und die Rangfolge der Kritik kann an zwei, drei Nennungen hängen (Sitzung 7).

✏️ **Teil 2: der Plausibilitätscheck.** Nehmt die Zusammenfassung für euren Ausschnitt und prüft sie **Satz für Satz**:
1. Lasst den Zahlen-Check laufen: `zahlen_check(text_teil, f_teil)`. Gibt es ein ❓?
2. Sucht **einen Satz ohne Zahl** und prüft ihn mit `belege(...)` gegen die Bewertungen, z. B. `belege("App & Bedienung", teil=teil)`.
3. Notiert: Welcher Satz war am schwersten zu prüfen, und warum?

Bringt eure Notiz nächste Woche mit: Dort geht es genau darum, wann das LLM lügt.

In [ ]:
# Platz für euren Plausibilitätscheck
zahlen_check(text_teil, f_teil)

## 8. Die Vorlage für eure Projektarbeit

Dieses Notebook ist das Gerüst, das ihr für eure eigenen Daten wiederverwendet. Was ihr austauscht, und was bleibt:

| Baustein | in diesem Notebook | in eurem Projekt |
|---|---|---|
| Daten | Bewertungen zur Nimbus Q2 | eure Daten, bereinigt wie in Sitzung 4 |
| Schema und Prompt | Stimmung, Themen, dringend | eure Felder und euer Katalog |
| Analysieren | `analysiere`, parallel | bleibt fast gleich |
| Prüfen | Vergleich mit der Wahrheit | Vergleich mit einer **selbst annotierten Stichprobe** |
| Kennzahlen und Dashboard | `kennzahlen`, `dashboard` | eure Fragen, eure vier Felder |
| Zusammenfassung | Fakten → LLM → Zahlen-Check | bleibt gleich |

Der Schritt, der am häufigsten fehlt, ist der Prüfschritt. Ohne eine annotierte Stichprobe wisst ihr nicht, ob euer Dashboard stimmt.

## Zusammenfassung

```text
1. Pipeline                            -> Bausteine als Funktionen: bereinigen, analysieren, prüfen,
                                          Kennzahlen, Dashboard, Zusammenfassung
2. Analysieren                         -> Schema + Korrektur (Sitzung 6), parallel + Retry (Sitzung 3)
3. Prüfen                              -> einzeln (Treffer, F1) UND im Aggregat: ändert sich die Antwort?
4. Dashboard                           -> vier Fragen, vier Felder; jede Zahl mit ihrer Einschränkung
5. Zusammenfassung                     -> LLM formuliert aus geprüften Fakten, rechnet nicht selbst
6. Zahlen-Check und Belege             -> jede Zahl und jede Behauptung auf eine Quelle zurückführen
```

## Zum Weiterprobieren (freiwillig)

1. **Ein anderer Ausschnitt:** das Dashboard nur für das dritte Quartal (`df["quartal"] == pd.Period("2026Q3")`). Wie ändert sich die Kritik-Rangliste?
2. **Zusammenfassung ohne Regeln:** Löscht die Regeln aus `SUMMARY_PROMPT` und lasst neu schreiben. Was ändert sich am Ton, was an den Zahlen? Lasst den Zahlen-Check laufen.
3. **Zusammenfassung aus Rohdaten:** Gebt dem LLM statt der Fakten 50 zufällige Bewertungen und lasst es zusammenfassen. Vergleicht mit dem Dashboard: Wo weicht es ab?
4. **Ein fünftes Feld:** Welche Frage würde Lena als Nächstes stellen? Baut ein fünftes Feld ins Dashboard.

## Ausblick

Damit ist die erste Hälfte des Kurses geschafft: Ihr habt ein vollständiges Insight-Werkzeug gebaut, von den Rohdaten bis zum Absatz für den Chef.

**Nächste Woche (02.12):** Wir drehen den Spieß um. Nicht mehr *bauen*, sondern *misstrauen*: **Wann lügt das LLM?** Bringt eure Notiz aus dem Plausibilitätscheck mit.

**Eure Projektarbeit:** Der Prüfungsbrief (Version 1) steht ab heute auf Moodle. Fragen dazu stellt ihr im Moodle-Forum; wir beantworten sie am Ende der Sitzungen am 02.12, 09.12 und 16.12. Bis zum **16.12** reicht jede Gruppe Domäne und Datenquelle ein.